In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    matthews_corrcoef,
    make_scorer,
    f1_score,
)
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedGroupKFold,
    StratifiedKFold,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

In [ ]:
DIR_PATH = '/content/drive/MyDrive/1. Academics/ENEE408N/ENEE408N Project/sample of DREAMT dataset'
# DIR_PATH = '/data'
TRAIN_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_train.csv')
TEST_PATH = os.path.join(DIR_PATH, 'unbal_all_ppg_feat_extr_test.csv')

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

_drop_train = ['Label']
_drop_test  = ['Label']

if 'patient_id' in train_df.columns:
    groups_train = train_df['patient_id'].values
    groups_test  = test_df['patient_id'].values
    _drop_train  = ['Label', 'patient_id']
    _drop_test   = ['Label', 'patient_id']
else:
    groups_train = None
    groups_test  = None

X_train = train_df.drop(_drop_train, axis=1)
y_train = train_df['Label']
X_test  = test_df.drop(_drop_test, axis=1)
y_test  = test_df['Label']

A = sum(y_train)
X = len(y_train)
print(f'Apnea count: {A}/{X} ({100*A/X:.2f}%)')
print(f'Nonapnea count: {X-A}/{X} ({100*(X-A)/X:.2f}%)')
print(f'patient_id present: {groups_train is not None}')

# ML Models

In [ ]:
def select_scaler(model, dim_reduction):
    """Return appropriate scaler based on model and dim_reduction types."""
    if isinstance(dim_reduction, SelectKBest) and dim_reduction.score_func is chi2:
        return MinMaxScaler()
    if isinstance(model, (KNeighborsClassifier, SVC)):
        return StandardScaler()
    return None  # RandomForest and others need no scaling


def build_pipeline(model, scaler, dim_reduction, oversampler):
    """Build an ImbPipeline dynamically without if/else branching per combination.

    Steps order: oversampler -> (scaler?) -> (dim_reduction?) -> model
    """
    steps = [("oversampler", oversampler)]
    if scaler is not None:
        steps.append(("scaler", scaler))
    if dim_reduction is not None:
        steps.append(("dim_reduction", dim_reduction))
    steps.append(("model", model))
    return ImbPipeline(steps)


def param_grid(model_name, dim_reduction_name):
    """Return hyperparameter search space keyed by pipeline step name."""
    grid = {}

    model_grids = {
        "knn": {"model__n_neighbors": [3, 5, 7, 11]},
        "rf": {
            "model__n_estimators": [50, 100, 200],
            "model__max_depth": [None, 5, 10, 20],
        },
        "svm": {
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", "auto", 0.01, 0.001],
        },
    }
    grid.update(model_grids.get(model_name, {}))

    dim_reduction_grids = {
        "selectkbest_chi2": {"dim_reduction__k": [5, 10, 15, 20]},
        "selectkbest_f_classif": {"dim_reduction__k": [5, 10, 15, 20]},
        "pca": {"dim_reduction__n_components": [3, 5, 10, 15]},
    }
    grid.update(dim_reduction_grids.get(dim_reduction_name, {}))

    return grid


def model_report(pipeline, X_test, y_test, show=False):
    y_pred = pipeline.predict(X_test)

    metrics = {}
    metrics['accuracy'] = accuracy_score(y_test, y_pred)
    metrics['conf_mat'] = confusion_matrix(y_test, y_pred)
    metrics['classification_rep'] = classification_report(y_test, y_pred)

    try:
        metrics['proba_scores'] = pipeline.predict_proba(X_test)
    except AttributeError:
        metrics['proba_scores'] = None

    if show:
        print("Accuracy:", metrics['accuracy'])
        print("Confusion matrix:\n", metrics['conf_mat'])
        print("Classification report:\n", metrics['classification_rep'])

    return metrics


def run_experiment(X, y, groups, model, param_grid_dict, scaler, dim_reduction):
    """Nested patient-level CV with threshold tuning.

    Outer: StratifiedGroupKFold(n_splits=5) — no patient leaks across splits.
    Inner: GridSearchCV scored on MCC.
    Threshold: per fold, sweep [0,1] and pick lowest FPR meeting >=80% sensitivity.

    Returns dict:
        best_params        – list of best params per outer fold
        median_cv_threshold – median of per-fold operating thresholds
        fold_metrics       – list of dicts with sensitivity/specificity/mcc/threshold per fold
    """
    if hasattr(X, 'reset_index'):
        X = X.reset_index(drop=True)
    if hasattr(y, 'reset_index'):
        y = y.reset_index(drop=True)

    sgkf = StratifiedGroupKFold(n_splits=5)
    mcc_scorer = make_scorer(matthews_corrcoef)
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    fold_thresholds = []
    fold_metrics = []
    best_params_list = []

    for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
        X_fold_tr = X.iloc[train_idx] if hasattr(X, 'iloc') else X[train_idx]
        X_fold_val = X.iloc[val_idx] if hasattr(X, 'iloc') else X[val_idx]
        y_fold_tr = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]
        y_fold_val = y.iloc[val_idx] if hasattr(y, 'iloc') else y[val_idx]

        pipeline = build_pipeline(
            clone(model),
            clone(scaler) if scaler is not None else None,
            clone(dim_reduction) if dim_reduction is not None else None,
            RandomOverSampler(random_state=42),
        )

        search = GridSearchCV(
            pipeline,
            param_grid_dict,
            scoring=mcc_scorer,
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
        )
        search.fit(X_fold_tr, y_fold_tr)
        best_params_list.append(search.best_params_)

        proba = search.best_estimator_.predict_proba(X_fold_val)[:, 1]

        # Sweep thresholds: pick lowest FPR that meets >=80% sensitivity
        best_thresh = 0.5
        best_fpr = float('inf')
        for t in np.linspace(0.0, 1.0, 101):
            y_pred_t = (proba >= t).astype(int)
            cm = confusion_matrix(y_fold_val, y_pred_t, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            if sens >= 0.80 and fpr < best_fpr:
                best_fpr = fpr
                best_thresh = float(t)

        fold_thresholds.append(best_thresh)

        y_pred_val = (proba >= best_thresh).astype(int)
        cm = confusion_matrix(y_fold_val, y_pred_val, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        fold_metrics.append({
            'fold': fold_idx,
            'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            'mcc': matthews_corrcoef(y_fold_val, y_pred_val),
            'threshold': best_thresh,
            'best_params': search.best_params_,
        })

    return {
        'best_params': best_params_list,
        'median_cv_threshold': float(np.median(fold_thresholds)),
        'fold_metrics': fold_metrics,
    }


def evaluate_on_test(pipeline, X_test, y_test, threshold):
    """Apply calibrated threshold to held-out test set exactly once.

    Returns sensitivity, MCC, specificity, precision, apnea-class F1, AUC-ROC.
    """
    proba  = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (proba >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'precision':   tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'mcc':         matthews_corrcoef(y_test, y_pred),
        'f1_apnea':    f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        'auc_roc':     roc_auc_score(y_test, proba),
    }

## KNN

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    name='knn_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='knn_3_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + Chi_square

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='knn_3_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='knn_3_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

KNN + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=KNeighborsClassifier(n_neighbors=3),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='knn_3_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

## RF

RF + Univariate

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='rf_100w_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + chi

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='rf_100w_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + PCA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='rf_100w_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

RF + LDA

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_leaf=5
)
pipeline = model_pipeline(
    X_train,
    y_train,
    model=rf,
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='rf_100w_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

In [ ]:
import os
# Check it exists and see its size
path = os.path.join(DIR_PATH, 'rf_100w_lda_1.joblib')
print(os.path.exists(path))        # should be True
print(os.path.getsize(path), "bytes")

# Download
from google.colab import files
files.download(path)

## SVM

SVM + univariate

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=SelectKBest(k=5),
    name='svm_rbf_univariate_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + chi2

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=MinMaxScaler(),
    dim_reduction=SelectKBest(score_func=chi2, k=5),
    name='svm_rbf_chi_5'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + PCA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=PCA(n_components=3),
    name='svm_rbf_pca_3'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

SVM + LDA

In [ ]:
pipeline = model_pipeline(
    X_train,
    y_train,
    model=SVC(kernel='rbf'),
    scaler=StandardScaler(),
    dim_reduction=LinearDiscriminantAnalysis(n_components=1),
    name='svm_rbf_lda_1'
  )
metrics = model_report(pipeline, X_test, y_test, show=True)

# Pipeline Helpers Demo & Tests

In [ ]:
# Demo: KNN + SelectKBest(chi2) end-to-end through build_pipeline
_demo_dim_red = SelectKBest(score_func=chi2, k=5)
_demo_model = KNeighborsClassifier(n_neighbors=3)
_demo_oversampler = RandomOverSampler(random_state=42)
_demo_scaler = select_scaler(_demo_model, _demo_dim_red)

_demo_pipeline = build_pipeline(
    model=_demo_model,
    scaler=_demo_scaler,
    dim_reduction=_demo_dim_red,
    oversampler=_demo_oversampler,
)
_demo_pipeline.fit(X_train, y_train)
print("Demo pipeline steps:", [name for name, _ in _demo_pipeline.steps])
_ = model_report(_demo_pipeline, X_test, y_test, show=True)

In [ ]:
# Tests: verify build_pipeline step names and order
def _test_build_pipeline():
    ros = RandomOverSampler(random_state=0)
    knn = KNeighborsClassifier()
    svm = SVC()
    rf = RandomForestClassifier()
    ss = StandardScaler()
    mm = MinMaxScaler()
    chi_sel = SelectKBest(score_func=chi2, k=5)
    f_sel = SelectKBest(k=5)
    pca = PCA(n_components=3)

    # Case 1: oversampler only + model -> 2 steps
    p = build_pipeline(rf, None, None, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "model"], f"Case1 fail: {p.steps}"

    # Case 2: oversampler + scaler + model -> 3 steps
    p = build_pipeline(knn, ss, None, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "scaler", "model"], f"Case2 fail: {p.steps}"

    # Case 3: oversampler + dim_reduction + model -> 3 steps
    p = build_pipeline(rf, None, pca, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "dim_reduction", "model"], f"Case3 fail: {p.steps}"

    # Case 4: all present -> 4 steps
    p = build_pipeline(knn, ss, pca, ros)
    assert [n for n, _ in p.steps] == ["oversampler", "scaler", "dim_reduction", "model"], f"Case4 fail: {p.steps}"

    # select_scaler: chi2 -> MinMaxScaler
    assert isinstance(select_scaler(knn, chi_sel), MinMaxScaler), "chi2 scaler fail"

    # select_scaler: KNN + non-chi2 -> StandardScaler
    assert isinstance(select_scaler(knn, f_sel), StandardScaler), "knn scaler fail"

    # select_scaler: SVM -> StandardScaler
    assert isinstance(select_scaler(svm, f_sel), StandardScaler), "svm scaler fail"

    # select_scaler: RF -> None
    assert select_scaler(rf, f_sel) is None, "rf scaler fail"

    # param_grid contains expected keys
    g = param_grid("knn", "pca")
    assert "model__n_neighbors" in g, "knn param missing"
    assert "dim_reduction__n_components" in g, "pca param missing"

    g = param_grid("rf", "selectkbest_chi2")
    assert "model__n_estimators" in g
    assert "model__max_depth" in g
    assert "dim_reduction__k" in g

    g = param_grid("svm", "selectkbest_f_classif")
    assert "model__C" in g
    assert "model__gamma" in g
    assert "dim_reduction__k" in g

    print("All pipeline helper tests passed.")

_test_build_pipeline()

In [ ]:
# Tests: nested CV, evaluate_on_test, and threshold sensitivity guarantee

def _test_no_patient_leakage():
    """No patient appears in both train and val of any outer fold."""
    np.random.seed(0)
    n_patients = 20
    reps = 10
    patient_ids = np.repeat(np.arange(n_patients), reps)
    X_syn = pd.DataFrame(np.random.rand(n_patients * reps, 4))
    y_syn = pd.Series((patient_ids % 2).astype(int))

    sgkf = StratifiedGroupKFold(n_splits=5)
    for train_idx, val_idx in sgkf.split(X_syn, y_syn, patient_ids):
        train_pats = set(patient_ids[train_idx])
        val_pats   = set(patient_ids[val_idx])
        assert train_pats.isdisjoint(val_pats), \
            f"Leakage: patients {train_pats & val_pats} in both splits"
    print("No patient leakage: PASSED")


def _test_evaluate_on_test_hand_calc():
    """evaluate_on_test metrics match hand-calculated values on fixed proba array."""
    # proba >= 0.5 -> preds: [1,1,0,0,1,1,0,0]
    # y_true:              [1,1,0,0,1,0,0,1]
    # TP=3(idx 0,1,4), FP=1(idx 5), TN=3(idx 2,3,6), FN=1(idx 7)
    # sensitivity=3/4=0.75, specificity=3/4=0.75, precision=3/4=0.75
    _proba = np.array([0.9, 0.8, 0.3, 0.2, 0.7, 0.6, 0.1, 0.4])
    _y_true = np.array([1,   1,   0,   0,   1,   0,   0,   1])

    class _FixedPipeline:
        def predict_proba(self, X):
            return np.column_stack([1 - _proba, _proba])

    X_dummy = np.zeros((len(_proba), 1))
    res = evaluate_on_test(_FixedPipeline(), X_dummy, _y_true, threshold=0.5)

    assert abs(res['sensitivity'] - 0.75) < 1e-9, f"sensitivity={res['sensitivity']}"
    assert abs(res['specificity'] - 0.75) < 1e-9, f"specificity={res['specificity']}"
    assert abs(res['precision']   - 0.75) < 1e-9, f"precision={res['precision']}"
    print("evaluate_on_test hand-calc: PASSED")


def _test_run_experiment_sensitivity():
    """run_experiment returns per-fold thresholds achieving >=80% sensitivity."""
    np.random.seed(42)
    n_patients = 30
    reps = 6
    patient_ids = np.repeat(np.arange(n_patients), reps)
    patient_labels = np.array([i % 2 for i in range(n_patients)])
    labels = np.repeat(patient_labels, reps)

    X_syn = pd.DataFrame({
        'signal': labels + np.random.normal(0, 0.05, len(labels)),
        'noise':  np.random.rand(len(labels)),
    })
    y_syn = pd.Series(labels.astype(int))

    result = run_experiment(
        X_syn, y_syn, patient_ids,
        model=RandomForestClassifier(n_estimators=10, random_state=42),
        param_grid_dict={"model__n_estimators": [10]},
        scaler=None,
        dim_reduction=None,
    )

    for fm in result['fold_metrics']:
        assert fm['sensitivity'] >= 0.80, \
            f"Fold {fm['fold']} sensitivity={fm['sensitivity']:.3f} < 0.80"
    print(f"run_experiment sensitivity (>=0.80 per fold): PASSED  "
          f"median_threshold={result['median_cv_threshold']:.3f}")


_test_no_patient_leakage()
_test_evaluate_on_test_hand_calc()
_test_run_experiment_sensitivity()

In [ ]:
# Demo: Nested CV + final test eval — KNN + PCA (test set touched exactly once)

_demo_model      = KNeighborsClassifier()
_demo_scaler     = StandardScaler()
_demo_dim_red    = PCA(n_components=5)
_demo_pg         = param_grid("knn", "pca")

print("=== Nested CV (outer=StratifiedGroupKFold-5, inner=GridSearchCV-MCC) ===")
_ncv = run_experiment(
    X_train, y_train, groups_train,
    model=_demo_model,
    param_grid_dict=_demo_pg,
    scaler=_demo_scaler,
    dim_reduction=_demo_dim_red,
)

print(f"Median CV threshold : {_ncv['median_cv_threshold']:.4f}\n")
_fold_df = pd.DataFrame(_ncv['fold_metrics'])[['fold', 'sensitivity', 'specificity', 'mcc', 'threshold']]
print(_fold_df.to_string(index=False))

# Final model: retrain on all X_train with inner CV to select best hyperparams
print("\n=== Final model (trained on full X_train) ===")
_final_pipeline = build_pipeline(
    KNeighborsClassifier(), StandardScaler(), PCA(),
    RandomOverSampler(random_state=42),
)
_final_search = GridSearchCV(
    _final_pipeline, _demo_pg,
    scoring=make_scorer(matthews_corrcoef),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1, refit=True,
)
_final_search.fit(X_train, y_train)
print(f"Best params: {_final_search.best_params_}")

# Evaluate on test set exactly once using the CV-calibrated threshold
print("\n=== Final Test Metrics (test set touched once) ===")
_test_result = evaluate_on_test(
    _final_search.best_estimator_, X_test, y_test, _ncv['median_cv_threshold']
)
print(pd.Series(_test_result).to_string())

In [ ]:
from sklearn.feature_selection import f_classif

# ── Experiment matrix: 3 models × 4 dim-reductions = 12 combinations ──────────

_MODELS = {
    "KNN":          (KNeighborsClassifier(), "knn"),
    "RandomForest": (RandomForestClassifier(random_state=42), "rf"),
    "SVM":          (SVC(kernel="rbf", probability=True, random_state=42), "svm"),
}

_DIM_REDUCTIONS = {
    "SelectKBest_f_classif": SelectKBest(f_classif, k=10),
    "SelectKBest_chi2":      SelectKBest(chi2, k=10),
    "PCA":                   PCA(n_components=10),
    "LDA":                   LinearDiscriminantAnalysis(n_components=1),
}

results_rows = []

for model_name, (model_obj, model_key) in _MODELS.items():
    for dr_name, dr_obj in _DIM_REDUCTIONS.items():
        print(f"\n>>> {model_name} + {dr_name}")

        scaler = select_scaler(model_obj, dr_obj)

        # LDA: binary classification → n_components fixed at 1, skip dim-red param tuning
        if dr_name == "LDA":
            pg = param_grid(model_key, "lda")   # returns only model params
        else:
            dr_key = "selectkbest_f_classif" if dr_name == "SelectKBest_f_classif" else \
                     "selectkbest_chi2"       if dr_name == "SelectKBest_chi2" else \
                     "pca"
            pg = param_grid(model_key, dr_key)

        # Nested CV → median threshold + fold metrics
        cv_result = run_experiment(
            X_train, y_train, groups_train,
            model=model_obj,
            param_grid_dict=pg,
            scaler=scaler,
            dim_reduction=clone(dr_obj),
        )
        median_thresh = cv_result["median_cv_threshold"]

        # Retrain final model on all X_train
        final_pipe = build_pipeline(
            clone(model_obj),
            clone(scaler) if scaler is not None else None,
            clone(dr_obj),
            RandomOverSampler(random_state=42),
        )
        final_search = GridSearchCV(
            final_pipe, pg,
            scoring=make_scorer(matthews_corrcoef),
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
            n_jobs=-1, refit=True,
        )
        final_search.fit(X_train, y_train)
        final_estimator = final_search.best_estimator_

        # Evaluate on test set exactly once
        test_metrics = evaluate_on_test(final_estimator, X_test, y_test, median_thresh)

        row = {
            "model":            model_name,
            "dim_reduction":    dr_name,
            "sensitivity":      test_metrics["sensitivity"],
            "mcc":              test_metrics["mcc"],
            "specificity":      test_metrics["specificity"],
            "precision":        test_metrics["precision"],
            "f1_apnea":         test_metrics["f1_apnea"],
            "auc_roc":          test_metrics["auc_roc"],
            "cv_threshold":     median_thresh,
            "_pipeline":        final_estimator,   # kept for export; dropped later
        }
        results_rows.append(row)
        print(f"    sensitivity={row['sensitivity']:.3f}  mcc={row['mcc']:.3f}  "
              f"auc={row['auc_roc']:.3f}  thresh={median_thresh:.3f}")

print("\nAll 12 combinations complete.")

In [ ]:
# ── Naive baseline + best-model export + final comparison DataFrame ───────────

# Naive baseline: always predict non-apnea (class 0)
# Sensitivity = 0%, Specificity = 100%, Accuracy ≈ 81% (dataset imbalance)
_apnea_rate = float(y_test.mean())
naive_accuracy = 1.0 - _apnea_rate   # ~0.81 for this dataset

naive_row = {
    "model":         "Naive (always non-apnea)",
    "dim_reduction": "—",
    "sensitivity":   0.0,
    "mcc":           0.0,
    "specificity":   1.0,
    "precision":     float("nan"),
    "f1_apnea":      0.0,
    "auc_roc":       float("nan"),
    "cv_threshold":  float("nan"),
}

# ── Pick best model: highest MCC; break ties by sensitivity ──────────────────
best_row = max(results_rows, key=lambda r: (r["mcc"], r["sensitivity"]))
best_pipeline = best_row["_pipeline"]

# Export best pipeline via joblib
_best_name = f"{best_row['model'].lower().replace(' ', '_')}_{best_row['dim_reduction'].lower()}"
_export_path = os.path.join(DIR_PATH, f"best_model_{_best_name}.joblib")
joblib.dump(best_pipeline, _export_path)
print(f"Best model exported → {_export_path}")
print(f"  ({best_row['model']} + {best_row['dim_reduction']}  "
      f"MCC={best_row['mcc']:.4f}  sensitivity={best_row['sensitivity']:.4f})")

# ── Build display DataFrame (drop internal pipeline column) ───────────────────
display_rows = [
    {k: v for k, v in r.items() if k != "_pipeline"}
    for r in results_rows
]
display_rows.append(naive_row)

results_df = pd.DataFrame(display_rows, columns=[
    "model", "dim_reduction",
    "sensitivity", "mcc", "specificity", "precision",
    "f1_apnea", "auc_roc", "cv_threshold",
])

# ── Style: flag sensitivity ≥ 80% clinical floor ─────────────────────────────
CLINICAL_FLOOR = 0.80

def _flag_sensitivity(val):
    """Bold + star rows that meet the ≥80% clinical sensitivity floor."""
    try:
        return "font-weight: bold; color: green;" if val >= CLINICAL_FLOOR else "color: red;"
    except TypeError:
        return ""

styled = (
    results_df.style
    .format({
        "sensitivity": "{:.1%}",
        "mcc":         "{:.4f}",
        "specificity": "{:.1%}",
        "precision":   "{:.1%}",
        "f1_apnea":    "{:.4f}",
        "auc_roc":     "{:.4f}",
        "cv_threshold": "{:.4f}",
    }, na_rep="N/A")
    .applymap(_flag_sensitivity, subset=["sensitivity"])
    .set_caption(
        f"Comparison table — 12 (model × dim-reduction) combinations + naive baseline. "
        f"Green sensitivity ≥ {CLINICAL_FLOOR:.0%} clinical floor."
    )
    .set_table_styles([{"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]}])
)

styled

# Feedforward Neural Net